In [1]:
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import datetime, timedelta
import os

In [4]:
fake = Faker("en_IN")
np.random.seed(42)
random.seed(42)

RAW_PATH = "netflix_titles.csv"
OUTPUT_PATH = "data/generated"

os.makedirs(OUTPUT_PATH, exist_ok=True)

In [5]:
content = pd.read_csv(RAW_PATH)

content["content_id"] = (
    content["show_id"]
    .str.replace("s", "", regex=False)
    .astype(int)
)

content_ids = content["content_id"].tolist()

In [6]:
NUM_USERS = 25000

users = []

for user_id in range(1, NUM_USERS + 1):
    signup_date = fake.date_between(
        start_date="-3y",
        end_date="-30d"
    )

    users.append({
        "user_id": user_id,
        "full_name": fake.name(),
        "gender": random.choice(["Male", "Female", "Other"]),
        "age": random.randint(18, 65),
        "city": fake.city(),
        "state": fake.state(),
        "country": "India",
        "signup_date": signup_date,
        "preferred_language": random.choice(
            ["Hindi", "English", "Bengali", "Tamil", "Telugu", "Malayalam"]
        ),
        "user_segment": random.choice(
            ["Light User", "Regular User", "Power User"]
        )
    })

users_df = pd.DataFrame(users)
users_df.to_csv(f"{OUTPUT_PATH}/users.csv", index=False)

In [7]:
devices = []

device_id = 1

for user_id in users_df["user_id"]:
    num_devices = random.randint(1, 3)

    for _ in range(num_devices):
        devices.append({
            "device_id": device_id,
            "user_id": user_id,
            "device_type": random.choice(["Mobile", "TV", "Laptop", "Tablet"]),
            "os": random.choice(["Android", "iOS", "Windows", "macOS", "Smart TV OS"]),
            "app_version": random.choice(["1.0.1", "1.2.0", "2.0.0", "2.1.5"]),
            "registered_date": fake.date_between(
                start_date="-3y",
                end_date="today"
            )
        })

        device_id += 1

devices_df = pd.DataFrame(devices)
devices_df.to_csv(f"{OUTPUT_PATH}/devices.csv", index=False)

In [8]:
subscriptions = []

plans = {
    "Basic": 199,
    "Standard": 499,
    "Premium": 649
}

subscription_id = 1

for user_id in users_df["user_id"]:
    plan_type = random.choice(list(plans.keys()))
    start_date = users_df.loc[
        users_df["user_id"] == user_id, "signup_date"
    ].values[0]

    status = random.choices(
        ["Active", "Cancelled", "Expired"],
        weights=[0.65, 0.20, 0.15]
    )[0]

    if status == "Active":
        end_date = None
    else:
        end_date = fake.date_between(
            start_date=pd.to_datetime(start_date),
            end_date="today"
        )

    subscriptions.append({
        "subscription_id": subscription_id,
        "user_id": user_id,
        "plan_type": plan_type,
        "start_date": start_date,
        "end_date": end_date,
        "status": status,
        "monthly_price": plans[plan_type],
        "billing_cycle": "Monthly"
    })

    subscription_id += 1

subscriptions_df = pd.DataFrame(subscriptions)
subscriptions_df.to_csv(f"{OUTPUT_PATH}/subscriptions.csv", index=False)

In [9]:
payments = []

payment_id = 1

for _, row in subscriptions_df.iterrows():
    start_date = pd.to_datetime(row["start_date"])
    months_active = random.randint(1, 24)

    for i in range(months_active):
        payment_date = start_date + pd.DateOffset(months=i)

        if payment_date > pd.Timestamp.today():
            break

        payments.append({
            "payment_id": payment_id,
            "user_id": row["user_id"],
            "subscription_id": row["subscription_id"],
            "payment_date": payment_date.date(),
            "amount": row["monthly_price"],
            "payment_method": random.choice(
                ["UPI", "Credit Card", "Debit Card", "Net Banking", "Wallet"]
            ),
            "payment_status": random.choices(
                ["Success", "Failed", "Refunded"],
                weights=[0.88, 0.10, 0.02]
            )[0]
        })

        payment_id += 1

payments_df = pd.DataFrame(payments)
payments_df.to_csv(f"{OUTPUT_PATH}/payments.csv", index=False)

In [10]:
NUM_SESSIONS = 200000

sessions = []

for session_id in range(1, NUM_SESSIONS + 1):
    user_id = random.randint(1, NUM_USERS)

    user_devices = devices_df[devices_df["user_id"] == user_id]["device_id"].tolist()
    device_id = random.choice(user_devices)

    session_start = fake.date_time_between(
        start_date="-2y",
        end_date="now"
    )

    duration = random.randint(5, 180)
    session_end = session_start + timedelta(minutes=duration)

    sessions.append({
        "session_id": session_id,
        "user_id": user_id,
        "device_id": device_id,
        "session_start": session_start,
        "session_end": session_end,
        "session_duration_minutes": duration,
        "traffic_source": random.choice(
            ["Direct", "Search", "Social Media", "Email", "Recommendation"]
        )
    })

sessions_df = pd.DataFrame(sessions)
sessions_df.to_csv(f"{OUTPUT_PATH}/sessions.csv", index=False)

In [11]:
watch_history = []

watch_id = 1

for _, session in sessions_df.iterrows():
    num_watches = random.choices(
        [1, 2, 3, 4],
        weights=[0.55, 0.30, 0.10, 0.05]
    )[0]

    for _ in range(num_watches):
        content_id = random.choice(content_ids)

        watch_duration = random.randint(5, min(180, session["session_duration_minutes"]))
        completion_percentage = random.randint(10, 100)

        watch_history.append({
            "watch_id": watch_id,
            "user_id": session["user_id"],
            "content_id": content_id,
            "session_id": session["session_id"],
            "watch_start_time": session["session_start"],
            "watch_duration_minutes": watch_duration,
            "completion_percentage": completion_percentage,
            "is_completed": completion_percentage >= 85,
            "watch_device": devices_df.loc[
                devices_df["device_id"] == session["device_id"],
                "device_type"
            ].values[0]
        })

        watch_id += 1

watch_history_df = pd.DataFrame(watch_history)
watch_history_df.to_csv(f"{OUTPUT_PATH}/watch_history.csv", index=False)

In [12]:
events = []

event_id = 1
event_types = ["play", "pause", "resume", "search", "like", "dislike", "add_to_watchlist"]

for _, row in watch_history_df.sample(300000, random_state=42).iterrows():
    events.append({
        "event_id": event_id,
        "user_id": row["user_id"],
        "session_id": row["session_id"],
        "content_id": row["content_id"],
        "event_time": row["watch_start_time"],
        "event_type": random.choice(event_types),
        "event_metadata": random.choice([
            '{"source":"home"}',
            '{"source":"recommendation"}',
            '{"source":"search"}',
            '{"quality":"HD"}',
            '{"quality":"4K"}'
        ])
    })

    event_id += 1

events_df = pd.DataFrame(events)
events_df.to_csv(f"{OUTPUT_PATH}/user_events.csv", index=False)

In [13]:
print("Data generation completed successfully!")

print("Users:", len(users_df))
print("Devices:", len(devices_df))
print("Subscriptions:", len(subscriptions_df))
print("Payments:", len(payments_df))
print("Sessions:", len(sessions_df))
print("Watch History:", len(watch_history_df))
print("User Events:", len(events_df))

Data generation completed successfully!
Users: 25000
Devices: 50189
Subscriptions: 25000
Payments: 252813
Sessions: 200000
Watch History: 329790
User Events: 300000
